# Stage 3: Direct Preference Optimization (DPO)
This notebook implements Direct Preference Optimization (DPO) on top of the SFT model, utilizing preference pairs (prompt, chosen, rejected) to fine-tune the output tone, accuracy, and safety constraints.

In [1]:
# Install libraries in Google Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft transformers accelerate bitsandbytes
!pip install unsloth_zoo
!pip install datasets
!pip install trl


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jqwrmtsg/unsloth_aa3a5c882cba487ba4e70353aa0266d1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jqwrmtsg/unsloth_aa3a5c882cba487ba4e70353aa0266d1
  Resolved https://github.com/unslothai/unsloth.git to commit d105bd7b42ea8d4ecdb3e3364abb605b558c017e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 140.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.5 MB/s eta 0:00:00
  

In [2]:
from google.colab import drive
import os
import shutil

# First, attempt to unmount the drive if it's currently mounted.
try:
    drive.flush_and_unmount()
    print("Drive unmounted successfully.")
except ValueError:
    print("Drive not mounted, so nothing to flush and unmount.")
    pass

# Ensure the mount point directory is empty.
# If /content/drive exists and is not empty, it will cause the ValueError.
mount_point = '/content/drive'
if os.path.isdir(mount_point):
    print(f"Cleaning up existing directory: {mount_point}")
    # Remove all contents of the directory
    for filename in os.listdir(mount_point):
        file_path = os.path.join(mount_point, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')
else:
    print(f"Creating mount point directory: {mount_point}")
    os.makedirs(mount_point, exist_ok=True)

# Now, mount the drive.
# force_remount=True will handle cases where the directory might have residual
# mount information or if a previous mount was incomplete.
drive.mount(mount_point, force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Drive unmounted successfully.
Creating mount point directory: /content/drive
Mounted at /content/drive


In [3]:
import torch
from unsloth import FastLanguageModel


max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load SFT model for preference alignment
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,

)

# Apply LoRA for DPO
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
from datasets import load_dataset

# Load the preference dataset (upload 'preference_dataset.jsonl' via the sidebar)
#dataset = load_dataset("json", data_files="preference_dataset.jsonl", split="train")
dataset = load_dataset("json", data_files="/content/drive/MyDrive/AIML-2026/preference_dataset.jsonl", split="train")


# Format inputs for DPO
prompt_format = """Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

def format_dpo(examples):
    formatted = {
        "prompt": [],
        "chosen": [],
        "rejected": []
    }
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        formatted["prompt"].append(prompt_format.format(instruction=prompt))
        formatted["chosen"].append(chosen + tokenizer.eos_token)
        formatted["rejected"].append(rejected + tokenizer.eos_token)
    return formatted

dataset = dataset.map(format_dpo, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [5]:
from trl import DPOTrainer
from transformers import TrainingArguments

# Initialize TrainingArguments
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    max_steps = 50,
    learning_rate = 5e-6,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.0,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = "dpo_outputs",
    remove_unused_columns = False,
)

# Manually add attributes to TrainingArguments for Unsloth's DPOTrainer
training_args.model_init_kwargs = None
training_args.ref_model_init_kwargs = None
training_args.padding_value = tokenizer.pad_token_id
training_args.generate_during_eval = False
training_args.model_adapter_name = None
training_args.ref_adapter_name = None
training_args.reference_free = False
training_args.disable_dropout = False
training_args.use_liger_loss = False
training_args.label_pad_token_id = tokenizer.pad_token_id
training_args.max_prompt_length = 512
training_args.max_completion_length = 512
training_args.max_length = 1024
training_args.truncation_mode = "longest_first"
training_args.precompute_ref_log_probs = False
training_args.use_logits_to_keep = False
training_args.padding_free = False
training_args.beta = 0.1
training_args.label_smoothing = 0.0
training_args.loss_type = "sigmoid"
training_args.loss_weights = None
training_args.use_weighting = False
training_args.f_divergence_type = "kl"
training_args.f_alpha_divergence_coef = 0.0
training_args.dataset_num_proc = None
training_args.tools = None
training_args.sync_ref_model = False
training_args.rpo_alpha = None
training_args.ld_alpha = None # Add this line to fix the new error

# Initialize DPOTrainer using Unsloth wrapper configuration
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth automatically handles saving memory by freezing baseline weights
    args = training_args, # Use the modified training_args
    beta = 0.1, # Implicit reward factor for preference comparison
    train_dataset = dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

dpo_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/content/unsloth_compiled_cache/UnslothDPOTrainer.py:1027: UserWarning: The `padding_value` argument is deprecated and will be removed in version 0.26.0. Please use `pad_token` (str) instead.
  warnings.warn(


Extracting prompt in train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55 | Num Epochs = 8 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.692571,0.000134,-0.001027,0.325000,0.001161,-101.790306,-77.812096,-0.933144,-0.573695
10,0.685123,0.003003,-0.012971,0.925000,0.015974,-95.330040,-76.963387,-0.964426,-0.618885
15,0.664518,0.008868,-0.047849,1.000000,0.056717,-101.540161,-77.027756,-1.014419,-0.675391
20,0.635236,0.016112,-0.103707,1.000000,0.119819,-97.049202,-78.018616,-0.853401,-0.524055
25,0.601882,0.023437,-0.173654,1.000000,0.197091,-101.340057,-79.464645,-1.120698,-0.732720
30,0.581033,0.032070,-0.212566,1.000000,0.244636,-98.339119,-81.658920,-0.895290,-0.593996
35,0.554601,0.042159,-0.255703,1.000000,0.297862,-100.797211,-83.446739,-1.001064,-0.646401
40,0.543456,0.046087,-0.282136,1.000000,0.328223,-99.637039,-80.939552,-0.951117,-0.601763
45,0.533060,0.027864,-0.330667,1.000000,0.358531,-96.561424,-83.756798,-1.040362,-0.737150
50,0.533128,0.055501,-0.291337,1.000000,0.346838,-97.303528,-78.623398,-0.958892,-0.616724


Unsloth: Restored added_tokens_decoder metadata in dpo_outputs/checkpoint-50/tokenizer_config.json.


TrainOutput(global_step=50, training_loss=0.6024609422683715, metrics={'train_runtime': 116.9097, 'train_samples_per_second': 3.421, 'train_steps_per_second': 0.428, 'total_flos': 0.0, 'train_loss': 0.6024609422683715, 'epoch': 7.142857142857143})

In [6]:
from unsloth import FastLanguageModel

# Inference test on DPO model
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    prompt_format.format(
        instruction = "What is the difference between the Free Plan and the Pro Plan?",
        response = "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 150, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
What is the difference between the Free Plan and the Pro Plan?

### Response:
The Free Plan is free for students and includes 10 practice speeches, while the Pro Plan costs $12 per month and grants 50 practice speeches.


In [7]:
# Save DPO aligned model locally
model.save_pretrained("dpo_aligned_model")
tokenizer.save_pretrained("dpo_aligned_model")

Unsloth: Restored added_tokens_decoder metadata in dpo_aligned_model/tokenizer_config.json.


('dpo_aligned_model/tokenizer_config.json', 'dpo_aligned_model/tokenizer.json')

In [8]:

# 1. Merge DPO adapter into your SFT-merged base and save it to Google Drive
model.save_pretrained_merged("/content/drive/MyDrive/AIML-2026/qwen2.5-7b-dpo-final-merged", tokenizer, save_method="merged_16bit")



Detected local model directory: /content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/AIML-2026/qwen2.5-7b-dpo-final-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:10<00:00, 10.79s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:14<00:00, 14.05s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/AIML-2026/qwen2.5-7b-dpo-final-merged`


In [11]:
from google.colab import userdata
from huggingface_hub import login, upload_folder, create_repo
import os

# 2. Push that local merged folder directly to Hugging Face
# Ensure your Hugging Face token is stored as a Colab secret named 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')

# Now you can use hf_token in your login function or other operations
login(token=hf_token)
print("Logged in to Hugging Face successfully!")

# The merged model was saved in the previous cell at this path
local_merged_model_path = "/content/drive/MyDrive/AIML-2026/qwen2.5-7b-dpo-final-merged"

hf_repo_name = "Bhargav1/qwen2.5-7b-speech-dpo"

print(f"Uploading model to Hugging Face Hub as {hf_repo_name}...")

# Create the repository on Hugging Face Hub if it doesn't exist
# This is required for upload_folder to work correctly
create_repo(repo_id=hf_repo_name, private=True, exist_ok=True, token=hf_token)

# Upload the folder containing the merged model directly using huggingface_hub
upload_folder(
    repo_id=hf_repo_name,
    folder_path=local_merged_model_path,
    token=hf_token,
    commit_message="Upload merged DPO model",
)

print("Upload Complete!")

Logged in to Hugging Face successfully!
Uploading model to Hugging Face Hub as Bhargav1/qwen2.5-7b-speech-dpo...
Upload Complete!
